# Projeto Fictus | Analise de Vendas — Bloco 2: Qualidade do Motor de Receita

---

## Pergunta Central do Bloco
> **O motor de receita é robusto ou perigosamente concentrado?**

---

## Contexto do Bloco

O Bloco 1 confirmou crescimento consistente da empresa-alvo, com sinais de pressão sobre margem via frete e possível erosão de valor por transação. A próxima etapa da diligência é testar a qualidade estrutural dessa receita: de onde ela vem, quão diversificada é e se resistiria a um movimento competitivo concentrado.

Um negócio pode crescer e ainda ser frágil se toda a receita vier de poucos produtos ou categorias. Um comprador que herda concentração herda também risco de colapso localizado — um único movimento competitivo pode destruir a tese de aquisição.

**Este bloco investiga:**
1. De onde exatamente vem a receita da empresa?
2. A receita está excessivamente concentrada em poucos produtos, clientes ou canais?
3. Os principais vendedores são estáveis ao longo do tempo ou há alta rotatividade?
4. Como o mix de produtos evoluiu ao longo do tempo?
5. Existem produtos ou categorias que aumentam receita, mas reduzem margem?
6. O ticket médio por categoria está crescendo ou se commoditizando?
7. O efeito Pareto se intensifica ou se dilui com o crescimento?
8. Há indícios de canibalização entre categorias ou canais?
9. A receita por vendedor está se concentrando nos mesmos ao longo do tempo — ou há renovação saudável da base de sellers ativos?

---

## Nota Metodológica — Deslocamento Temporal
Dados originais Olist **2016–2018** deslocados **+7 anos** → período **2023–2025**.  
**Análise restrita a partir de janeiro/2024** — dados de 2023 desconsiderados.

---

## Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from pathlib import Path

# ─── Caminhos relativos — funcionam em qualquer máquina ─────────────────────
# O notebook está em notebooks/ → a raiz é um nível acima
NOTEBOOK_DIR = Path().resolve()
# Detecta a raiz do projeto subindo a hierarquia de pastas
# Funciona em qualquer estrutura: raiz/, notebooks/, notebooks/vendas/
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(NOTEBOOK_DIR)
DIR_PRE      = BASE_DIR / "data" / "pre-tratados"
DIR_EXT      = BASE_DIR / "data" / "externos"
DIR_EXPORTS  = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)


warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")
print("Ambiente configurado.")

## Carregamento dos Dados

In [ ]:
# ─── Separador padrão: vírgula, decimal ponto ────────────────────────────────
#   Todos os arquivos gerados pelo ETL usam sep=',' e decimal='.'
def ler_csv(caminho, sep=",", **kwargs):
    df = pd.read_csv(caminho, sep=sep, low_memory=False, **kwargs)
    df.columns = df.columns.str.strip()
    return df

fato  = ler_csv(DIR_PRE / "fato_vendas.csv")
dim_p = ler_csv(DIR_PRE / "dim_produto.csv")
dim_c = ler_csv(DIR_PRE / "dim_cliente.csv")
dim_v = ler_csv(DIR_PRE / "dim_vendedor.csv")
dim_t = ler_csv(DIR_PRE / "dim_tempo.csv")

# ─── Conversão de datas ───────────────────────────────────────────────────────
for col in ["data_compra", "data_entrega_cliente", "data_previsao_entrega",
            "data_envio_transportadora", "data_aprovacao"]:
    if col in fato.columns:
        fato[col] = pd.to_datetime(fato[col], errors="coerce")
dim_t["data"] = pd.to_datetime(dim_t["data"], errors="coerce")

# ─── Colunas numéricas: garantia adicional ────────────────────────────────────
for col in ["preco", "valor_frete", "valor_total_item", "valor_pagamento_total",
            "lead_time_dias", "atraso_dias", "nota_review", "numero_parcelas"]:
    if col in fato.columns:
        fato[col] = pd.to_numeric(fato[col], errors="coerce")

# ─── Enriquecimento e filtro ─────────────────────────────────────────────────
fato = fato.merge(dim_p[["id_produto", "nome_categoria_produto"]], on="id_produto",  how="left")
fato = fato.merge(dim_c[["id_cliente", "estado_cliente"]],         on="id_cliente",  how="left")
fato = fato.merge(dim_v[["id_vendedor", "estado_vendedor"]],        on="id_vendedor", how="left")
fato = fato.merge(
    dim_t[["id_data", "ano", "mes", "trimestre", "nome_mes", "ano_mes"]],
    on="id_data", how="left"
)
fato["periodo"] = fato["ano"].astype(str) + "-Q" + fato["trimestre"].astype(str)

DATA_INICIO = "2024-01-01"
fato = fato[fato["data_compra"] >= DATA_INICIO].copy()

fe  = fato[fato["status_pedido"] == "entregue"].copy()
fne = fato[fato["status_pedido"] != "entregue"].copy()
periodos_ord = sorted(fato["periodo"].dropna().unique())

print(f"[FILTRO] Dados a partir de: {DATA_INICIO}")
print(f"fato (filtrado)   : {len(fato):>8} linhas | {fato['data_compra'].min().date()} → {fato['data_compra'].max().date()}")
print(f"fato_entregues    : {len(fe):>8} linhas ({len(fe)/len(fato)*100:.1f}% do total)")
print(f"Categorias únicas : {fe['nome_categoria_produto'].nunique()}")
print(f"Produtos únicos   : {fe['id_produto'].nunique()}")
print(f"Trimestres        : {len(periodos_ord)} — {periodos_ord[0]} → {periodos_ord[-1]}")


---

## Análise 1 — De onde exatamente vem a receita da empresa?

> *"A segmentação da receita por produto e categoria, fundamentada no Princípio de Pareto, permite identificar os pilares reais do faturamento. Em gestão de portfólio, é comum que poucos elementos expliquem a maior parte do resultado operacional. Esta decomposição é essencial para transcender a visão agregada e compreender a estrutura fundamental do motor de receita, isolando o que é core do que é periférico."*

**Framework:** Análise de Pareto (80/20)  
**Entrega:** Curva de Pareto por categoria com ranking das categorias que sustentam 80% do faturamento

**Como este script responde à pergunta:**
> A pergunta exige sair da receita total e descer ao nível de categoria. O script agrupa todos os pedidos entregues por categoria, soma a receita de cada uma, ordena do maior para o menor e calcula dois indicadores centrais: quantas categorias são necessárias para atingir 80% da receita (`n_cats_80`) e quanto o top 20% do portfólio representa no faturamento total (`receita_top20p`). Com esses números em mãos, constrói dois gráficos complementares:
>
> 1. **Curva de Pareto:** As barras mostram o percentual de receita de cada categoria. As azuis são o top 20% do portfólio; as cinzas, o restante. A linha laranja acima acumula esses percentuais — quando ela cruza a linha vermelha dos 80%, temos o ponto exato de concentração. Quanto mais íngreme a curva, mais concentrada é a receita em poucas categorias.
> 2. **Ranking do top 20%:** O gráfico da direita isola apenas as categorias do top 20% e as ordena horizontalmente com seu percentual exato. Transforma o Pareto abstrato em uma lista de prioridades — quem sustenta o faturamento e quem é coadjuvante.

> **Critério de avaliação de concentração:** considera-se saudável quando o top 20% das categorias representa entre 60% e 85% da receita — concentração relevante, mas sem dependência extrema. Acima de 90% em poucas categorias configura risco estrutural crítico para a decisão de aquisição.

**Análise do Resultado:**
 Identificar de onde vem a receita é o primeiro passo para entender a fragilidade do portfólio. Se poucas categorias sustentam a maior parte do faturamento, qualquer movimento competitivo ou mudança de comportamento do consumidor nesse núcleo tem impacto imediato e desproporcional no resultado — risco que o comprador precisa conhecer antes de fechar o negócio.

In [ ]:
pareto_cat = (
    fe.groupby("nome_categoria_produto")
    .agg(receita=("preco", "sum"), n_itens=("id_pedido", "count"))
    .reset_index()
    .sort_values("receita", ascending=False)
    .reset_index(drop=True)
)
pareto_cat["pct_receita"] = pareto_cat["receita"] / pareto_cat["receita"].sum() * 100
pareto_cat["pct_acum"]    = pareto_cat["pct_receita"].cumsum()
pareto_cat["rank"]        = pareto_cat.index + 1

n_cats_80      = (pareto_cat["pct_acum"] <= 80).sum() + 1
top20p_cats    = int(np.ceil(len(pareto_cat) * 0.2))
receita_top20p = pareto_cat.head(top20p_cats)["pct_receita"].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Análise 1 — De onde vem a receita? Pareto de Categorias", fontsize=13, fontweight="bold")

top_n     = min(30, len(pareto_cat))
cores_bar = [COR_RECEITA if i < top20p_cats else COR_NEUTRO for i in range(top_n)]
axes[0].bar(range(top_n), pareto_cat.head(top_n)["pct_receita"], color=cores_bar, alpha=0.85)
ax1_twin = axes[0].twinx()
ax1_twin.plot(range(top_n), pareto_cat.head(top_n)["pct_acum"],
              color=COR_DESTAQUE, linewidth=2, marker="o", markersize=3)
ax1_twin.axhline(80, color=COR_ALERTA, linestyle="--", linewidth=1, alpha=0.7)
ax1_twin.set_ylabel("% Receita Acumulada", color=COR_DESTAQUE)
ax1_twin.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_ylabel("% Receita Individual")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_xlabel("Categorias (ordenadas por receita)")
axes[0].set_title(f"Curva de Pareto — Top {top_n} Categorias", fontsize=11)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_RECEITA, label=f"Top 20% ({top20p_cats} cats)"),
    mpatches.Patch(color=COR_NEUTRO,  label="Demais"),
], frameon=False, fontsize=8)

cats_label = pareto_cat.head(top20p_cats)["nome_categoria_produto"].tolist()
receitas   = pareto_cat.head(top20p_cats)["pct_receita"].tolist()
bars2 = axes[1].barh(
    [c.replace("_", " ") for c in cats_label],
    receitas,
    color=[plt.cm.Blues_r(i / len(cats_label)) for i in range(len(cats_label))],
    alpha=0.9
)
for bar, val in zip(bars2, receitas):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{val:.1f}%", va="center", fontsize=8)
axes[1].set_xlabel("% da Receita Total")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title(f"Top 20% das Categorias ({top20p_cats} de {len(pareto_cat)})", fontsize=11)

plt.tight_layout()
salvar(fig, "01_pareto_categorias")
plt.show()

print(f"\nTotal de categorias: {len(pareto_cat)} | Top 20% = {receita_top20p:.1f}% da receita | {n_cats_80} cats para 80%")

---

## Análise 2 — A receita está excessivamente concentrada em poucos produtos ou canais?

> *"A concentração excessiva representa uma vulnerabilidade estratégica que reduz a resiliência do negócio diante de movimentos competitivos. A aplicação da análise de Pareto com foco em risco estrutural permite avaliar se o portfólio possui dependências que poderiam comprometer a continuidade do faturamento em cenários de instabilidade setorial ou perda de parceiros-chave."*

**Framework:** Pareto — análise de concentração e risco estrutural  
**Entrega:** Pareto de produtos individuais e de vendedores, com curvas de concentração

**Como este script responde à pergunta:**
> Concentração de receita em poucas categorias é uma coisa. Concentração em poucos produtos individuais e poucos vendedores é outra — e muito mais crítica. O script calcula Pareto em dois níveis distintos, cada um com sua curva acumulada e seu ponto de 80%:
>
> 1. **Concentração por produto:** Agrupa receita por `id_produto` e plota a curva acumulada sobre os produtos ordenados do maior para o menor. A linha pontilhada vertical marca onde estão os produtos que somam 80% da receita, com rótulo automático mostrando quantos produtos isso representa e qual percentual do catálogo total eles ocupam. Quanto mais à esquerda a linha, mais perigosa a dependência.
> 2. **Concentração por vendedor:** Aplica a mesma lógica sobre os sellers, calculando também o número de pedidos únicos, categorias atendidas e nota média de cada um. A curva acumulada de sellers revela se o risco de concentração vem do portfólio de produtos ou da base de fornecedores — ou dos dois simultaneamente.
>
> Os dois gráficos juntos formam o diagnóstico completo de risco estrutural: um negócio pode ter portfólio diversificado e ainda assim ser altamente dependente de um único seller que abastece várias categorias.

**Análise do Resultado:** O objetivo aqui é medir a "fragilidade" do portfólio. Um negócio onde 80% da receita vem de apenas dois ou três produtos é um negócio de alto risco; se esses produtos saírem de moda ou forem copiados, a empresa quebra. Buscamos um equilíbrio onde a receita seja distribuída, garantindo que a queda de uma peça não derrube o tabuleiro inteiro.


In [ ]:
pareto_prod = (
    fe.groupby("id_produto")
    .agg(
        receita      = ("preco",       "sum"),
        frete_total  = ("valor_frete", "sum"),
        n_vendas     = ("id_pedido",   "count"),
        nota_media   = ("nota_review", "mean"),
        categoria    = ("nome_categoria_produto", "first"),
    )
    .reset_index().sort_values("receita", ascending=False).reset_index(drop=True)
)
pareto_prod["pct_receita"] = pareto_prod["receita"] / pareto_prod["receita"].sum() * 100
pareto_prod["pct_acum"]    = pareto_prod["pct_receita"].cumsum()
pareto_prod["pct_frete"]   = pareto_prod["frete_total"] / pareto_prod["receita"] * 100

pareto_seller = (
    fe.groupby("id_vendedor")
    .agg(
        receita      = ("preco",                 "sum"),
        n_pedidos    = ("id_pedido",              "nunique"),
        n_categorias = ("nome_categoria_produto", "nunique"),
        nota_media   = ("nota_review",            "mean"),
        n_trimestres = ("periodo",                "nunique"),
    )
    .reset_index().sort_values("receita", ascending=False).reset_index(drop=True)
)
pareto_seller["pct_receita"] = pareto_seller["receita"] / pareto_seller["receita"].sum() * 100
pareto_seller["pct_acum"]    = pareto_seller["pct_receita"].cumsum()

total_produtos    = len(pareto_prod)
n_prods_80        = (pareto_prod["pct_acum"] <= 80).sum() + 1
pct_prods_80      = n_prods_80 / total_produtos * 100
total_sellers     = len(pareto_seller)
n_sellers_80      = (pareto_seller["pct_acum"] <= 80).sum() + 1
pct_sellers_80    = n_sellers_80 / total_sellers * 100
top10_sellers_pct = pareto_seller.head(10)["pct_receita"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 2 — Concentração: Produtos e Vendedores", fontsize=13, fontweight="bold")

n_plot = min(500, total_produtos)
axes[0].plot(range(n_plot), pareto_prod.head(n_plot)["pct_acum"], color=COR_RECEITA, linewidth=2)
axes[0].axhline(80, color=COR_ALERTA, linestyle="--", linewidth=1)
axes[0].axvline(n_prods_80, color=COR_DESTAQUE, linestyle=":", linewidth=1)
axes[0].annotate(
    f"{n_prods_80} produtos\n= 80% receita\n({pct_prods_80:.1f}% do catálogo)",
    xy=(n_prods_80, 80), xytext=(n_prods_80 + max(5, n_plot*0.05), 65),
    fontsize=8, color=COR_DESTAQUE,
    arrowprops=dict(arrowstyle="->", color=COR_DESTAQUE, lw=1)
)
axes[0].set_xlabel(f"Produtos (de {total_produtos} únicos)")
axes[0].set_ylabel("% Receita Acumulada")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_title("Concentração de Receita — Produtos Individuais", fontsize=11)

n_plot_s = min(200, total_sellers)
axes[1].plot(range(n_plot_s), pareto_seller.head(n_plot_s)["pct_acum"], color=COR_RECEITA, linewidth=2)
axes[1].axhline(80, color=COR_ALERTA, linestyle="--", linewidth=1)
axes[1].axvline(n_sellers_80, color=COR_DESTAQUE, linestyle=":", linewidth=1)
axes[1].annotate(
    f"{n_sellers_80} sellers\n= 80% receita\n({pct_sellers_80:.1f}%)",
    xy=(n_sellers_80, 80), xytext=(n_sellers_80 + max(3, n_plot_s*0.08), 60),
    fontsize=8, color=COR_DESTAQUE,
    arrowprops=dict(arrowstyle="->", color=COR_DESTAQUE, lw=1)
)
axes[1].set_xlabel(f"Vendedores (de {total_sellers})")
axes[1].set_ylabel("% Receita Acumulada")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Concentração de Receita — Vendedores", fontsize=11)

plt.tight_layout()
salvar(fig, "02_concentracao")
plt.show()

print(f"\nProdutos para 80%: {n_prods_80} ({pct_prods_80:.1f}%) | Sellers para 80%: {n_sellers_80} ({pct_sellers_80:.1f}%) | Top 10 sellers: {top10_sellers_pct:.1f}%")

---

## Análise 3 — Os principais vendedores são estáveis ao longo do tempo ou há alta rotatividade?

> *"A análise da estabilidade longitudinal dos parceiros comerciais distingue dependências críticas estruturais de picos oportunistas de faturamento. Um vendedor com participação relevante e constante indica consolidação, enquanto novos entrantes com alto volume exigem validação de sustentabilidade. Esta distinção é vital para que o processo de aquisição identifique a perenidade dos fluxos de receita gerados por terceiros."*

**Framework:** Pareto + PDCA — análise de estabilidade e dependência crítica  
**Entrega:** Scatter de relevância × estabilidade dos top sellers, com classificação automática de sellers críticos
**Como este script responde à pergunta:**
> Saber que um seller representa 10% da receita não é suficiente — é preciso saber se ele está lá todo trimestre ou se apareceu recentemente. O script cruza dois cálculos: o percentual de receita de cada seller (já calculado na análise anterior) e o percentual de trimestres em que ele esteve ativo. A combinação define automaticamente quem é um seller crítico: qualquer um com mais de 1% da receita e presença em mais de 50% dos trimestres.
>
> 1. **Scatter relevância × estabilidade:** Cada ponto é um dos top 50 sellers. O eixo horizontal mostra em quantos trimestres ele esteve ativo; o eixo vertical mostra seu peso na receita. Os pontos vermelhos são sellers críticos — relevantes e recorrentes. O tamanho de cada ponto indica o número de categorias atendidas: sellers que operam em mais categorias têm maior poder de barganha e maior impacto em caso de saída.
> 2. **Qualidade dos top 20 sellers:** Ordena os 20 maiores sellers pela nota média de review, com o percentual de receita e de trimestres ativos anotados em cada barra. Sellers com nota abaixo de 3,5 são um risco de reputação ativo — já estão deteriorando a experiência do cliente independentemente de qualquer negociação futura.

**Análise do Resultado:** Esta métrica avalia a saúde do ecossistema de parceiros (sellers). Sellers estáveis e recorrentes indicam parceiros satisfeitos — mas quando poucos deles concentram parcela relevante da receita, a estabilidade vira dependência. O risco não está na rotatividade alta, mas na fossilização: um marketplace cujas posições de topo não renovam está refém de quem já está dentro, sem poder de barganha para renegociar condições no pós-aquisição.


In [ ]:
n_trimestres_total = fe["periodo"].nunique()
top50_ids = pareto_seller.head(50)["id_vendedor"].tolist()

estab = (
    fe[fe["id_vendedor"].isin(top50_ids)]
    .groupby(["id_vendedor", "periodo"])["preco"].sum()
    .reset_index()
    .groupby("id_vendedor")
    .agg(n_trim_ativo=("periodo", "nunique"))
    .reset_index()
)
pareto_seller = pareto_seller.merge(estab, on="id_vendedor", how="left")
pareto_seller["pct_trim_ativo"] = pareto_seller["n_trim_ativo"] / n_trimestres_total * 100
pareto_seller["seller_critico"] = (
    (pareto_seller["pct_receita"]   >= 1.0) &
    (pareto_seller["pct_trim_ativo"] >= 50)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Análise 3 — Estabilidade dos Principais Vendedores ao Longo do Tempo",
             fontsize=13, fontweight="bold")

# Scatter relevância × estabilidade
top50_plot = pareto_seller.head(50).dropna(subset=["pct_trim_ativo"])
cores_crit = [COR_ALERTA if c else COR_RECEITA for c in top50_plot["seller_critico"]]
axes[0].scatter(
    top50_plot["pct_trim_ativo"], top50_plot["pct_receita"],
    c=cores_crit, s=top50_plot["n_categorias"] * 20 + 20,
    alpha=0.8, edgecolors="white", linewidth=0.5
)
axes[0].axhline(1.0, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5, label="Limiar relevância: 1%")
axes[0].axvline(50,  color=COR_NEUTRO, linestyle=":",  linewidth=1, alpha=0.5, label="Limiar estabilidade: 50%")
xlim, ylim = axes[0].get_xlim(), axes[0].get_ylim()
axes[0].text(75, ylim[1]*0.85, "SELLER CRÍTICO\n(relevante + recorrente)",
             color=COR_ALERTA, fontsize=8, ha="center", alpha=0.7)
axes[0].set_xlabel("% Trimestres Ativo")
axes[0].set_ylabel("% da Receita Total")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_title("Top 50 Sellers: Relevância × Estabilidade\n(tamanho = nº categorias atendidas)", fontsize=11)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_ALERTA,  label="Seller crítico"),
    mpatches.Patch(color=COR_RECEITA, label="Demais"),
], frameon=False, fontsize=8)

# Nota média dos top sellers
top20_sellers = pareto_seller.head(20).sort_values("nota_media")
cores_nota = [COR_ALERTA if v < 3.5 else COR_DESTAQUE if v < 4.0 else COR_MARGEM
              for v in top20_sellers["nota_media"]]
axes[1].barh(range(len(top20_sellers)), top20_sellers["nota_media"], color=cores_nota, alpha=0.85)
axes[1].axvline(4.0, color=COR_NEUTRO, linestyle="--", linewidth=1, label="Benchmark 4.0")
axes[1].set_yticks(range(len(top20_sellers)))
axes[1].set_yticklabels(
    [f"#{int(r['rank'] if 'rank' in r else i+1)} | {r['pct_receita']:.1f}% | {int(r['pct_trim_ativo'] if pd.notna(r['pct_trim_ativo']) else 0)}% trim."
     for i, (_, r) in enumerate(top20_sellers.iterrows())],
    fontsize=7
)
axes[1].set_xlabel("Nota Média de Review")
axes[1].set_xlim(0, 5.2)
axes[1].set_title("Qualidade dos Top 20 Sellers\n(% receita | % trimestres ativo)", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "03_estabilidade_sellers")
plt.show()

n_criticos = pareto_seller["seller_critico"].sum()
pct_receita_criticos = pareto_seller[pareto_seller["seller_critico"]]["pct_receita"].sum()
print("\n" + "="*60)
print("INSIGHT — ESTABILIDADE DE SELLERS")
print("="*60)
print(f"Sellers críticos identificados   : {n_criticos} (>1% receita + >50% trimestres)")
print(f"Receita nos sellers críticos     : {pct_receita_criticos:.1f}%")
print(f"Risco: saída de um seller crítico pode impactar até {pareto_seller[pareto_seller['seller_critico']]['pct_receita'].max():.1f}% da receita diretamente")

---

## Análise 4 — Como o mix de produtos evoluiu ao longo do tempo?

> *"A observação da tendência evolutiva do portfólio, sob a ótica do ciclo de controle (PDCA), revela se a empresa está em processo de reposicionamento estratégico ou em uma trajetória de deterioração silenciosa da qualidade da receita. Mudanças graduais no mix são indicadores precoces de adaptabilidade do negócio às demandas de mercado ou de perda de relevância de categorias históricas."*

**Framework:** PDCA — monitoramento de tendência  
**Entrega:** Evolução da participação das top categorias por trimestre, em valor absoluto e participação relativa
**Como este script responde à pergunta:**
> Uma fotografia do mix diz o que o portfólio é hoje. Uma sequência de fotografias diz para onde ele está indo. O script calcula a receita de cada uma das top 8 categorias trimestre a trimestre e constrói dois pivôs: um com valores absolutos e outro com participação percentual de cada categoria no total do trimestre.
>
> 1. **Receita absoluta empilhada:** Cada barra é um trimestre com as categorias empilhadas em cores distintas. O crescimento da altura total confirma o crescimento geral — mas a composição das cores revela se todas as categorias crescem juntas ou se uma está dominando progressivamente o portfólio.
> 2. **Participação relativa (% do mix):** Normaliza todas as barras para 100%, eliminando o efeito do crescimento absoluto. Aqui ficam visíveis as mudanças estruturais que seriam invisíveis no gráfico absoluto: uma categoria pode crescer em reais e mesmo assim perder participação relativa se outras crescerem mais rápido — sinal de reposicionamento involuntário do portfólio.

**Análise do Resultado:** Observamos se a empresa está conseguindo inovar ou se está presa a produtos antigos. Um mix que evolui para categorias de maior valor agregado mostra maturidade. Se o mix está migrando para produtos mais baratos e simples, a empresa pode estar perdendo seu diferencial competitivo e entrando em uma zona de "guerra de preços".


In [ ]:
top8_cats = pareto_cat.head(8)["nome_categoria_produto"].tolist()

mix_temporal = fe.groupby(["periodo", "nome_categoria_produto"])["preco"].sum().reset_index()
pivot_abs = (
    mix_temporal[mix_temporal["nome_categoria_produto"].isin(top8_cats)]
    .pivot_table(index="periodo", columns="nome_categoria_produto", values="preco", aggfunc="sum")
    .fillna(0)
)
pivot_pct = pivot_abs.div(pivot_abs.sum(axis=1), axis=0) * 100

palette_mix = sns.color_palette("tab10", n_colors=len(top8_cats))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Análise 4 — Evolução do Mix de Categorias por Trimestre", fontsize=13, fontweight="bold")

pivot_abs.div(1000).plot(kind="bar", stacked=True, ax=axes[0], color=palette_mix, width=0.8, alpha=0.9)
axes[0].set_title("Receita Absoluta (R$ mil)", fontsize=11)
axes[0].set_ylabel("Receita (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}K"))
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(loc="upper left", fontsize=7, frameon=False)

pivot_pct.plot(kind="bar", stacked=True, ax=axes[1], color=palette_mix, width=0.8, alpha=0.9)
axes[1].set_title("Participação no Mix (%)", fontsize=11)
axes[1].set_ylabel("Participação (%)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend().remove()

plt.tight_layout()
salvar(fig, "04_mix_temporal")
plt.show()

primeiro = pivot_pct.iloc[0]
ultimo   = pivot_pct.iloc[-1]
variacao = (ultimo - primeiro).sort_values(ascending=False)

print("\n" + "="*60)
print(f"INSIGHT — EVOLUÇÃO DO MIX ({periodos_ord[0]} → {periodos_ord[-1]})")
print("="*60)
print("Categorias que ganharam participação:")
for cat, var in variacao[variacao > 0].items():
    print(f"  {cat:<38} +{var:.1f}pp")
print("Categorias que perderam participação:")
for cat, var in variacao[variacao < 0].items():
    print(f"  {cat:<38} {var:.1f}pp")

---

## Análise 5 — Existem categorias que aumentam receita, mas reduzem margem?

> *"O cruzamento entre crescimento de volume e qualidade financeira identifica distorções no equilíbrio do portfólio. Produtos que expandem o faturamento à custa da rentabilidade podem estar destruindo valor econômico. Esta análise de portfólio assegura que a tese de crescimento esteja alinhada à geração de caixa, evitando que o ativo cresça de forma insustentável."*

**Framework:** Análise de portfólio por crescimento e contribuição de margem   
**Entrega:** Quadrante crescimento × % frete (Q1: saudável, Q2: armadilha, Q3: desinvestimento limpo, Q4: deterioração)
  
**Como este script responde à pergunta:**
> Crescer em receita enquanto a eficiência piora é uma armadilha silenciosa. O script compara cada categoria em dois momentos: o primeiro e o último trimestre disponíveis. Para cada categoria com presença em pelo menos um terço dos trimestres, calcula dois deltas — a variação percentual de receita e a variação em pontos percentuais do % de frete sobre receita. Esses dois vetores definem a posição de cada categoria em um quadrante:
>
> 1. **Classificação automática em quadrantes:** Q1 (verde) — receita subiu e % frete caiu: crescimento saudável. Q2 (laranja) — receita subiu mas % frete também subiu: armadilha de crescimento. Q3 (cinza) — receita caiu mas % frete melhorou: desinvestimento limpo. Q4 (vermelho) — receita caiu e % frete piorou: deterioração total. O tamanho de cada bolha representa a receita atual da categoria no scatter — bolhas grandes no Q2 ou Q4 são os alertas mais críticos para a decisão de aquisição.
> 2. **Rótulos automáticos das maiores categorias:** As 12 categorias com maior receita no último período recebem rótulo de nome direto no gráfico, facilitando a identificação imediata de quais pilares do portfólio estão em cada quadrante sem precisar cruzar com outra tabela.

**Análise do Resultado:**
 É a identificação das 'falsas estrelas'. Uma categoria pode crescer em receita enquanto o frete como proporção dessa receita cresce ainda mais rápido — sinal de que o crescimento está comprimindo a eficiência financeira, não expandindo-a. Para o comprador, categorias no Q2 são as mais perigosas: crescem no relatório e deterioram silenciosamente nas margens.

In [ ]:
evolucao_cat = (
    fe.groupby(["periodo", "nome_categoria_produto"])
    .agg(receita=("preco", "sum"), frete_total=("valor_frete", "sum"))
    .reset_index()
)
evolucao_cat["pct_frete"] = evolucao_cat["frete_total"] / evolucao_cat["receita"] * 100

cats_validas = (
    evolucao_cat.groupby("nome_categoria_produto")["periodo"].nunique()
    .pipe(lambda s: s[s >= max(2, len(periodos_ord) // 3)].index)
)
primeiro_periodo = periodos_ord[0]
ultimo_periodo   = periodos_ord[-1]

m_inicio = evolucao_cat[evolucao_cat["periodo"] == primeiro_periodo].set_index("nome_categoria_produto")[["receita", "pct_frete"]]
m_fim    = evolucao_cat[evolucao_cat["periodo"] == ultimo_periodo].set_index("nome_categoria_produto")[["receita", "pct_frete"]]
comparativo = m_inicio.join(m_fim, lsuffix="_inicio", rsuffix="_fim", how="inner")
comparativo = comparativo[comparativo.index.isin(cats_validas)].copy()
comparativo["delta_receita_pct"] = (comparativo["receita_fim"] - comparativo["receita_inicio"]) / comparativo["receita_inicio"] * 100
comparativo["delta_frete_pp"]    = comparativo["pct_frete_fim"] - comparativo["pct_frete_inicio"]
comparativo = comparativo.reset_index().rename(columns={"nome_categoria_produto": "categoria"})

def quad(row):
    if   row["delta_receita_pct"] >= 0 and row["delta_frete_pp"] <= 0: return "Q1 — Crescimento saudável"
    elif row["delta_receita_pct"] >= 0 and row["delta_frete_pp"] >  0: return "Q2 — Armadilha de crescimento"
    elif row["delta_receita_pct"] <  0 and row["delta_frete_pp"] <= 0: return "Q3 — Desinvestimento limpo"
    else: return "Q4 — Deterioração total"

comparativo["quadrante"] = comparativo.apply(quad, axis=1)
paleta_q = {
    "Q1 — Crescimento saudável":     COR_MARGEM,
    "Q2 — Armadilha de crescimento": COR_DESTAQUE,
    "Q3 — Desinvestimento limpo":    COR_NEUTRO,
    "Q4 — Deterioração total":       COR_ALERTA,
}

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_title(
    f"Análise 5 — Portfólio: Crescimento × Pressão de Frete ({primeiro_periodo} → {ultimo_periodo})",
    fontsize=12, fontweight="bold"
)
for q, grupo in comparativo.groupby("quadrante"):
    ax.scatter(
        grupo["delta_receita_pct"], grupo["delta_frete_pp"],
        s=grupo["receita_fim"] / grupo["receita_fim"].max() * 600 + 30,
        color=paleta_q[q], alpha=0.75, label=f"{q} ({len(grupo)})",
        edgecolors="white", linewidth=0.5
    )
for _, row in comparativo.nlargest(12, "receita_fim").iterrows():
    ax.annotate(row["categoria"].replace("_", " "), (row["delta_receita_pct"], row["delta_frete_pp"]),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
xlim, ylim = ax.get_xlim(), ax.get_ylim()
ax.text(xlim[1]*0.55, ylim[1]*0.85, "Q1 — Cresce e melhora",   color=COR_MARGEM,   fontsize=9, alpha=0.7)
ax.text(xlim[1]*0.55, ylim[0]*0.85, "Q2 — Cresce mas pressiona", color=COR_DESTAQUE, fontsize=9, alpha=0.7)
ax.text(xlim[0]*0.90, ylim[0]*0.85, "Q4 — Deterioração total", color=COR_ALERTA,   fontsize=9, alpha=0.7)
ax.set_xlabel("Variação de Receita (%)")
ax.set_ylabel("Variação de % Frete (pp)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.1f}pp"))
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.tight_layout()
salvar(fig, "05_quadrante_portfolio")
plt.show()

n_q1 = len(comparativo[comparativo["quadrante"] == "Q1 — Crescimento saudável"])
n_q2 = len(comparativo[comparativo["quadrante"] == "Q2 — Armadilha de crescimento"])
n_q4 = len(comparativo[comparativo["quadrante"] == "Q4 — Deterioração total"])
print(f"\nQ1 Saudável: {n_q1} | Q2 Armadilha: {n_q2} | Q4 Deterioração: {n_q4}")

---

## Análise 6 — O ticket médio por categoria está crescendo ou se commoditizando?

> *"A queda do ticket médio em categorias com receita crescente sinaliza pressão competitiva ou perda de valor percebido, sugerindo uma transição para a competição por preço. Identificar quais pilares do portfólio sustentam valor e quais estão em processo de comoditização permite avaliar a força da marca e a margem de segurança contra novos entrantes."*

**Framework:** Controle de processo — Ciclo de Vida do Produto  
**Entrega:** Ticket médio por categoria com evolução temporal e identificação de categorias em commoditização
**Como este script responde à pergunta:**
> O script calcula o ticket médio (preço médio por item) de cada uma das top 10 categorias por receita, trimestre a trimestre. Em seguida constrói um pivô com categorias nas linhas e trimestres nas colunas, e calcula o delta de ticket entre o primeiro e o último período — tanto em reais quanto em percentual. O resultado é ordenado do maior para o menor queda, colocando as categorias em commoditização no topo da lista.
>
> 1. **Variação do ticket por categoria:** Barras horizontais mostram o delta em reais de cada categoria. Barras vermelhas (queda) identificam commoditização em curso; barras verdes (alta) indicam que o valor percebido se mantém ou cresce. O percentual de variação é anotado ao lado de cada barra para facilitar a comparação de magnitude entre categorias de preços muito diferentes.
> 2. **Evolução temporal das top 5:** Traça a curva do ticket médio trimestre a trimestre para as 5 maiores categorias. Linhas descendentes confirmam commoditização estrutural; linhas ascendentes ou estáveis indicam que o crescimento de volume não está corroendo o valor por transação. A leitura conjunta dos dois gráficos separa o diagnóstico pontual da tendência: uma categoria pode ter ticket baixo hoje mas estar em recuperação, ou ter ticket alto hoje mas em queda acelerada.

**Análise do Resultado:** O ticket médio é o termômetro do valor percebido. Se o valor gasto por compra está subindo, o cliente confia na marca e aceita pagar mais. Se o ticket médio cai consistentemente enquanto o volume sobe, a empresa está se "commoditizando" — tornando-se apenas uma opção barata, o que é uma posição perigosa e difícil de manter sem sacrificar a qualidade.


In [ ]:
ticket_cat = (
    fe.groupby(["periodo", "nome_categoria_produto"])
    .agg(ticket_medio=("preco", "mean"), n_itens=("id_pedido", "count"))
    .reset_index()
)

# Ticket médio no primeiro e último período
top10_cats_receita = pareto_cat.head(10)["nome_categoria_produto"].tolist()
ticket_pivot = (
    ticket_cat[ticket_cat["nome_categoria_produto"].isin(top10_cats_receita)]
    .pivot_table(index="nome_categoria_produto", columns="periodo", values="ticket_medio")
)
ticket_pivot["delta_ticket"] = ticket_pivot.iloc[:, -1] - ticket_pivot.iloc[:, 0]
ticket_pivot["pct_delta"]    = ticket_pivot["delta_ticket"] / ticket_pivot.iloc[:, 0] * 100
ticket_pivot = ticket_pivot.sort_values("delta_ticket", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Análise 6 — Ticket Médio por Categoria: Agregando ou Commoditizando?",
             fontsize=13, fontweight="bold")

# Variação de ticket
cores_tk = [COR_ALERTA if v < 0 else COR_MARGEM for v in ticket_pivot["delta_ticket"]]
axes[0].barh(
    [c.replace("_", " ")[:30] for c in ticket_pivot.index],
    ticket_pivot["delta_ticket"],
    color=cores_tk, alpha=0.85
)
axes[0].axvline(0, color="black", linewidth=0.8)
for i, (cat, row) in enumerate(ticket_pivot.iterrows()):
    axes[0].text(
        row["delta_ticket"] + (2 if row["delta_ticket"] >= 0 else -2),
        i, f"{row['pct_delta']:+.1f}%",
        va="center", ha="left" if row["delta_ticket"] >= 0 else "right",
        fontsize=8, color=COR_NEUTRO
    )
axes[0].set_xlabel(f"Variação do Ticket Médio (R$, {periodos_ord[0]} → {periodos_ord[-1]})")
axes[0].set_title("Variação do Ticket Médio por Categoria\n(negativo = commoditização)", fontsize=11)

# Evolução temporal do ticket nas top 5 categorias
palette_tk = sns.color_palette("tab10", n_colors=len(top10_cats_receita))
for i, cat in enumerate(top10_cats_receita[:5]):
    dados = ticket_cat[ticket_cat["nome_categoria_produto"] == cat].sort_values("periodo")
    if len(dados) > 0:
        axes[1].plot(range(len(dados)), dados["ticket_medio"].values,
                     marker="o", markersize=4, linewidth=1.5,
                     color=palette_tk[i], label=cat.replace("_", " ")[:25])
axes[1].set_xticks(range(len(periodos_ord)))
axes[1].set_xticklabels(periodos_ord, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Ticket Médio (R$)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}"))
axes[1].set_title("Evolução do Ticket Médio — Top 5 Categorias", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "06_ticket_por_categoria")
plt.show()

cats_commoditizando = ticket_pivot[ticket_pivot["delta_ticket"] < 0]
cats_agregando      = ticket_pivot[ticket_pivot["delta_ticket"] > 0]
print("\n" + "="*60)
print("INSIGHT — TICKET MÉDIO POR CATEGORIA")
print("="*60)
print(f"Categorias com ticket crescente (agregando valor)  : {len(cats_agregando)}")
print(f"Categorias com ticket decrescente (commoditizando) : {len(cats_commoditizando)}")
if len(cats_commoditizando):
    print("\nCategorias em commoditização (maior queda):")
    for cat, row in cats_commoditizando.head(3).iterrows():
        print(f"  {cat:<38} ticket: {row['pct_delta']:+.1f}%")

---

## Análise 7 — O efeito Pareto se intensifica ou se dilui com o crescimento?

> *"A análise da evolução da concentração de Pareto ao longo do tempo determina se a expansão do negócio é acompanhada de uma diversificação saudável ou de um aumento do risco de dependência. O monitoramento deste indicador serve para validar a robustez do motor de receita e a capacidade da empresa de escalar sem estreitar perigosamente sua base de sustentação."*

**Framework:** Pareto — análise dinâmica  
**Entrega:** Número de categorias necessárias para 80% da receita por trimestre, com detecção automática de tendência
**Como este script responde à pergunta:**
> A pergunta não é sobre concentração num momento — é sobre como a concentração evolui. O script define uma função que, para cada trimestre, calcula quantas categorias são necessárias para atingir 80% da receita daquele período. Aplica essa função em todos os trimestres e monta uma série temporal desse indicador, adicionando uma linha de tendência por regressão linear para confirmar a direção estatisticamente.
>
> 1. **Categorias para 80% da receita por trimestre:** Cada barra mostra o número de categorias necessárias naquele trimestre. Barras verdes indicam trimestres acima da mediana (maior diversificação); vermelhas, abaixo. A linha de tendência pontilhada e o texto automático no canto do gráfico dizem se o Pareto está diluindo (saudável) ou concentrando (risco).
> 2. **% da receita no top 20% das categorias:** Mede pelo outro lado: se o top 20% está capturando cada vez mais receita, a cauda longa está murchando. A área sombreada destaca o desvio em relação à média do período — azul quando o último ponto está abaixo da média (diversificação crescendo), vermelho quando está acima (concentração crescendo).

**Análise do Resultado:**
 Queremos saber se o crescimento está tornando a empresa mais dependente de poucos ("intensifica") ou se ela está conseguindo pulverizar suas vendas ("dilui"). Em M&A, um efeito Pareto que se dilui com o tempo é sinal de um negócio escalável e saudável, pois mostra que novos produtos e clientes estão ganhando relevância real.


In [ ]:
def cats_para_80pct(grupo):
    g = grupo.groupby("nome_categoria_produto")["preco"].sum().sort_values(ascending=False)
    pct_acum   = g.cumsum() / g.sum() * 100
    n          = (pct_acum < 80).sum() + 1
    total_cats = len(g)
    pct_top20  = g.head(max(1, int(np.ceil(total_cats * 0.2)))).sum() / g.sum() * 100
    return pd.Series({"n_cats_80pct": n, "total_cats": total_cats, "pct_top20_cats": pct_top20})

pareto_temporal = (
    fe.groupby("periodo").apply(cats_para_80pct)
    .reset_index().sort_values("periodo")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 7 — Efeito Pareto ao Longo do Tempo: Concentrando ou Diluindo?",
             fontsize=13, fontweight="bold")

x = range(len(pareto_temporal))
cores_n = [COR_MARGEM if v >= pareto_temporal["n_cats_80pct"].median() else COR_ALERTA
           for v in pareto_temporal["n_cats_80pct"]]
axes[0].bar(x, pareto_temporal["n_cats_80pct"], color=cores_n, alpha=0.85)
axes[0].axhline(pareto_temporal["n_cats_80pct"].mean(), color=COR_DESTAQUE, linestyle="--", linewidth=1.5,
                label=f"Média: {pareto_temporal['n_cats_80pct'].mean():.1f}")
if len(pareto_temporal) > 3:
    z = np.polyfit(range(len(pareto_temporal)), pareto_temporal["n_cats_80pct"], 1)
    axes[0].plot(x, np.poly1d(z)(list(x)), color="black", linewidth=1, linestyle=":", alpha=0.5)
    direcao = "↑ diluindo (saudável)" if z[0] > 0 else "↓ concentrando (risco)"
    axes[0].text(0.02, 0.95, f"Tendência: {direcao}", transform=axes[0].transAxes,
                 fontsize=9, color=COR_MARGEM if z[0] > 0 else COR_ALERTA)
axes[0].set_xticks(x)
axes[0].set_xticklabels(pareto_temporal["periodo"].tolist(), rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Nº de categorias")
axes[0].set_title("Categorias necessárias para 80% da receita", fontsize=11)
axes[0].legend(frameon=False)

axes[1].plot(x, pareto_temporal["pct_top20_cats"], color=COR_RECEITA, linewidth=2, marker="o", markersize=5)
axes[1].fill_between(
    x, pareto_temporal["pct_top20_cats"], pareto_temporal["pct_top20_cats"].mean(),
    alpha=0.15,
    color=COR_ALERTA if pareto_temporal["pct_top20_cats"].iloc[-1] > pareto_temporal["pct_top20_cats"].iloc[0]
    else COR_MARGEM
)
axes[1].axhline(80, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5, label="Referência 80%")
axes[1].set_xticks(x)
axes[1].set_xticklabels(pareto_temporal["periodo"].tolist(), rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("% Receita no Top 20% das Categorias")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("% Receita Concentrada no Top 20% das Categorias", fontsize=11)
axes[1].legend(frameon=False)

plt.tight_layout()
salvar(fig, "07_pareto_dinamico")
plt.show()

inicio     = pareto_temporal.iloc[0]
fim        = pareto_temporal.iloc[-1]
delta_cats = fim["n_cats_80pct"] - inicio["n_cats_80pct"]
direcao_pareto = "DILUIÇÃO (positivo)" if delta_cats > 0 else "CONCENTRAÇÃO (risco)" if delta_cats < 0 else "ESTÁVEL"
print(f"\n{inicio['periodo']}: {inicio['n_cats_80pct']:.0f} cats → {fim['periodo']}: {fim['n_cats_80pct']:.0f} cats — {direcao_pareto}")

---

## Análise 8 — Há indícios de canibalização entre categorias?

> *"Utilizando o pensamento sistêmico e a análise de interdependência, busca-se verificar se o crescimento de determinadas categorias ocorre de forma incremental ou se é apenas uma redistribuição interna de receita (canibalização). A correlação entre participações de mercado internas revela se a empresa está de fato expandindo sua atuação ou apenas deslocando faturamento entre seus próprios ativos."*

**Framework:** Análise sistêmica de interdependência — visão de causa e efeito  
**Entrega:** Heatmap de correlação de share entre top categorias — correlação negativa indica canibalização potencial
**Como este script responde à pergunta:**
> Para testar canibalização, o script precisa medir não receita absoluta, mas participação relativa — share. Calcula o share de cada categoria no total de cada trimestre, monta um pivô com trimestres nas linhas e categorias nas colunas, e aplica uma matriz de correlação de Pearson sobre esse pivô. Correlação negativa entre duas categorias significa que quando uma ganha share, a outra sistematicamente perde — evidência matemática de canibalização.
>
> 1. **Evolução do share das top 10 categorias:** Cada linha representa uma categoria mostrando sua participação percentual trimestre a trimestre. Linhas que sobem enquanto outras descem no mesmo período são o sinal visual de deslocamento de receita — uma categoria crescendo às custas de outra, não capturando demanda nova.
> 2. **Heatmap de correlação de share:** Quantifica o que o gráfico de linhas mostra visualmente. Células verdes indicam categorias que crescem juntas (expansão real do mercado); células vermelhas indicam pares que se canibalizam. O output de texto lista automaticamente os 5 pares com maior correlação negativa — os candidatos prioritários a investigação de causa raiz.

> **Nota metodológica:** correlação negativa entre shares indica associação inversa — não necessariamente competição direta entre categorias. A causalidade exige investigação complementar de sazonalidade, campanhas promocionais e perfil de cliente. Os pares listados são candidatos a investigação, não evidência conclusiva de canibalização.

**Análise do Resultado:** O crescimento de uma categoria não é necessariamente expansão de mercado — pode ser deslocamento interno de receita. Se o aumento de share de uma categoria coincide sistematicamente com a queda de outra, o portfólio está redistribuindo demanda em vez de capturá-la. Para o comprador, isso significa que o crescimento aparente de certas categorias pode não representar geração de valor nova — apenas realocação da mesma base de clientes.


In [ ]:
share_temporal = fe.groupby(["periodo", "nome_categoria_produto"])["preco"].sum().reset_index()
total_por_periodo = share_temporal.groupby("periodo")["preco"].sum().rename("total")
share_temporal = share_temporal.merge(total_por_periodo, on="periodo")
share_temporal["share"] = share_temporal["preco"] / share_temporal["total"] * 100

top10_cats  = pareto_cat.head(10)["nome_categoria_produto"].tolist()
share_top10 = share_temporal[share_temporal["nome_categoria_produto"].isin(top10_cats)]
pivot_share = share_top10.pivot_table(
    index="periodo", columns="nome_categoria_produto", values="share"
).fillna(0)
corr_shares = pivot_share.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Análise 8 — Canibalização entre Categorias (Top 10)", fontsize=13, fontweight="bold")

palette_cat = sns.color_palette("tab10", n_colors=len(top10_cats))
for i, cat in enumerate(top10_cats):
    dados = share_top10[share_top10["nome_categoria_produto"] == cat].sort_values("periodo")
    if len(dados) > 0:
        axes[0].plot(range(len(dados)), dados["share"].values,
                     marker="o", markersize=4, linewidth=1.5,
                     color=palette_cat[i], label=cat.replace("_", " ")[:25])
axes[0].set_xticks(range(len(periodos_ord)))
axes[0].set_xticklabels(periodos_ord, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Share de Receita (%)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_title("Evolução do Share das Top 10 Categorias", fontsize=11)
axes[0].legend(frameon=False, fontsize=7, loc="upper right", ncol=2)

short_names  = {c: c.replace("_", " ")[:18] for c in corr_shares.columns}
corr_display = corr_shares.rename(index=short_names, columns=short_names)
sns.heatmap(
    corr_display, ax=axes[1], annot=True, fmt=".2f", cmap="RdYlGn",
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    cbar_kws={"label": "Correlação de share"}, annot_kws={"size": 7}
)
axes[1].set_title("Correlação de Share\n(negativo = canibalização potencial)", fontsize=11)
axes[1].tick_params(axis="x", labelsize=7)
axes[1].tick_params(axis="y", labelsize=7)

plt.tight_layout()
salvar(fig, "08_canibalizacao")
plt.show()

pares = [
    {"cat1": corr_shares.columns[i], "cat2": corr_shares.columns[j],
     "correlacao": corr_shares.iloc[i, j]}
    for i in range(len(corr_shares.columns))
    for j in range(i+1, len(corr_shares.columns))
]
df_pares = pd.DataFrame(pares).sort_values("correlacao")
print("\nTop 5 pares com maior canibalização potencial:")
for _, r in df_pares.head(5).iterrows():
    print(f"  {r['cat1'][:28]:<30} × {r['cat2'][:28]:<30} corr: {r['correlacao']:.2f}")

---

## Análise 9 — A receita por vendedor está se concentrando nos mesmos ao longo do tempo — ou há renovação saudável da base de sellers ativos?

> *"A saúde de um ecossistema de vendas depende de uma renovação equilibrada da base de parceiros. A dependência excessiva de vendedores históricos cria uma "fossilização" que eleva o risco de renegociações desfavoráveis ou interrupções bruscas de faturamento. Analisar a rotatividade e a entrada de novos sellers permite distinguir uma base consolidada de uma estrutura frágil e dependente de poucos nomes."*

**Framework:** Análise de coorte de sellers + dinâmica de entrada e saída  
**Entrega:** Heatmap de participação de receita dos top sellers por trimestre + curva de renovação da base

**Como este script responde à pergunta:**
> A diferença entre um marketplace robusto e um frágil está na dinâmica da sua base de sellers: o primeiro renova constantemente, o segundo depende cada vez mais dos mesmos vendedores. Este script constrói duas visualizações para separar esses dois cenários:
>
> 1. **Heatmap de participação dos top sellers por trimestre:** Cada linha é um dos 20 maiores sellers (por receita total). As colunas são os trimestres analisados. A célula mostra qual % da receita total aquele seller representou naquele trimestre — células em azul escuro indicam sellers dominantes e estáveis; células brancas indicam sellers que sumiram ou ainda não chegaram. Se as mesmas linhas permanecem escuras durante todos os trimestres, a base está fossilizada — o crescimento depende de quem já está dentro, não de novos entrantes.
> 2. **Curva de renovação da base de sellers:** Para cada trimestre, calcula quantos sellers ativos são "novos" (não estavam presentes nos dois trimestres anteriores) e quantos são "veteranos". A linha de renovação mostra se a plataforma atrai consistentemente novos vendedores ou se a porta de entrada está fechada. Alta renovação com baixa dependência dos mesmos sellers é o sinal mais positivo para um comprador — indica que o marketplace tem poder de atração e não está refém de ninguém.

**Análise do Resultado:**
 Esta análise foca na meritocracia e renovação do marketplace. Se os mesmos vendedores dominam tudo há anos sem espaço para novos nomes, a base pode estar estagnada e resistente a mudanças. Uma renovação saudável prova que a plataforma atrai sangue novo e que o modelo de negócio é dinâmico o suficiente para sobreviver a trocas de gerações de vendedores.


In [ ]:
# ─── Análise 9 — Renovação vs Fossilização da Base de Sellers ────────────────

# Receita por seller por trimestre
seller_trim = (
    fe.groupby(["periodo", "id_vendedor"])
    .agg(receita=("preco", "sum"))
    .reset_index()
)
total_por_trim = seller_trim.groupby("periodo")["receita"].sum().rename("total")
seller_trim = seller_trim.merge(total_por_trim, on="periodo")
seller_trim["share"] = seller_trim["receita"] / seller_trim["total"] * 100

# Top 20 sellers por receita total
top20_sellers = (
    seller_trim.groupby("id_vendedor")["receita"].sum()
    .sort_values(ascending=False)
    .head(20)
    .index.tolist()
)

# Pivot para heatmap
periodos_ord = sorted(seller_trim["periodo"].unique())
heatmap_df = (
    seller_trim[seller_trim["id_vendedor"].isin(top20_sellers)]
    .pivot_table(index="id_vendedor", columns="periodo", values="share", fill_value=0)
)
# Ordenar por receita total
ordem = seller_trim[seller_trim["id_vendedor"].isin(top20_sellers)] \
    .groupby("id_vendedor")["receita"].sum().sort_values(ascending=False).index
heatmap_df = heatmap_df.reindex(ordem)
# Label curto para y-axis
heatmap_df.index = [f"Seller {i+1}" for i in range(len(heatmap_df))]

# ─── Curva de renovação ───────────────────────────────────────────────────────
sellers_por_trim = seller_trim.groupby("periodo")["id_vendedor"].apply(set).to_dict()
periodos_lista = sorted(sellers_por_trim.keys())

renovacao = []
for idx, p in enumerate(periodos_lista):
    ativos = sellers_por_trim[p]
    if idx < 2:
        renovacao.append({"periodo": p, "novos": len(ativos), "veteranos": 0, "pct_novos": 100})
        continue
    anteriores = sellers_por_trim[periodos_lista[idx-1]] | sellers_por_trim[periodos_lista[idx-2]]
    novos = ativos - anteriores
    vets  = ativos & anteriores
    pct_novos = len(novos) / len(ativos) * 100 if ativos else 0
    renovacao.append({"periodo": p, "novos": len(novos), "veteranos": len(vets), "pct_novos": pct_novos})

df_renov = pd.DataFrame(renovacao)
pct_novos_medio = df_renov["pct_novos"].iloc[2:].mean()

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Análise 9 — Renovação vs Fossilização da Base de Sellers", fontsize=13, fontweight="bold")

# Heatmap
im = axes[0].imshow(heatmap_df.values, aspect="auto", cmap="Blues", vmin=0)
plt.colorbar(im, ax=axes[0], label="% da receita total", fraction=0.03)
axes[0].set_xticks(range(len(heatmap_df.columns)))
axes[0].set_xticklabels([str(p) for p in heatmap_df.columns], rotation=45, ha="right", fontsize=8)
axes[0].set_yticks(range(len(heatmap_df.index)))
axes[0].set_yticklabels(heatmap_df.index, fontsize=8)
axes[0].set_xlabel("Trimestre")
axes[0].set_title("Share de Receita — Top 20 Sellers por Trimestre\n(azul escuro = dominância persistente)", fontsize=11)
for r in range(len(heatmap_df.index)):
    for c in range(len(heatmap_df.columns)):
        val = heatmap_df.values[r, c]
        if val > 0.3:
            axes[0].text(c, r, f"{val:.1f}%", ha="center", va="center",
                         fontsize=6, color="white" if val > 3 else "black")

# Curva de renovação (barras empilhadas + linha %)
x_r = range(len(df_renov))
axes[1].bar(x_r, df_renov["veteranos"], label="Sellers veteranos", color=COR_RECEITA, alpha=0.75)
axes[1].bar(x_r, df_renov["novos"], bottom=df_renov["veteranos"], label="Sellers novos", color=COR_MARGEM, alpha=0.85)
ax2 = axes[1].twinx()
ax2.plot(x_r, df_renov["pct_novos"], color=COR_DESTAQUE, linewidth=2, marker="o", markersize=4,
         label=f"% novos (média: {pct_novos_medio:.1f}%)")
ax2.axhline(pct_novos_medio, color=COR_DESTAQUE, linestyle="--", linewidth=1, alpha=0.5)
ax2.set_ylabel("% de sellers novos no trimestre", color=COR_DESTAQUE)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax2.tick_params(axis="y", labelcolor=COR_DESTAQUE)
axes[1].set_xticks(list(x_r))
axes[1].set_xticklabels([str(p) for p in df_renov["periodo"]], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Nº de sellers ativos")
axes[1].set_title("Composição da Base de Sellers por Trimestre\n(novos vs veteranos)", fontsize=11)
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=9)

plt.tight_layout()
salvar(fig, "09_renovacao_sellers")
plt.show()

# Métricas de concentração temporal
presenca_top20 = (heatmap_df > 0).sum(axis=1)
sellers_todos_trim = (presenca_top20 == len(periodos_ord)).sum()
seller1_share_medio = heatmap_df.iloc[0].mean()

print("\n" + "="*55)
print("INSIGHT — RENOVAÇÃO DA BASE DE SELLERS")
print("="*55)
print(f"Sellers presentes em TODOS os trimestres : {sellers_todos_trim} de {len(top20_sellers)}")
print(f"Share médio do top seller               : {seller1_share_medio:.1f}% da receita/trimestre")
print(f"% médio de sellers novos por trimestre  : {pct_novos_medio:.1f}%")
sinal_renov = "renovação saudável" if pct_novos_medio > 20 else "base fossilizada — baixa renovação"
print(f"Sinal de renovação                      : {sinal_renov}")


---

## Síntese do Bloco 2 — Qualidade do Motor de Receita
> **Limitações desta análise:** correlações de share entre categorias indicam associação estatística — não implicam causalidade ou competição direta comprovada. A análise de portfólio utiliza frete como proxy de margem, não margem líquida real. Investigação complementar é necessária antes de decisões operacionais.


In [ ]:
# ─── Veredicto 100% dinâmico — recalcula tudo localmente ────────────────────
# Concentração de categorias
_pareto_cat = (
    fe.groupby("nome_categoria_produto")
    .agg(receita=("preco", "sum"))
    .reset_index()
    .sort_values("receita", ascending=False)
    .reset_index(drop=True)
)
_pareto_cat["pct_receita"] = _pareto_cat["receita"] / _pareto_cat["receita"].sum() * 100
_pareto_cat["pct_acum"]    = _pareto_cat["pct_receita"].cumsum()
_n_cats_80      = (_pareto_cat["pct_acum"] <= 80).sum() + 1
_top20p_cats    = int(np.ceil(len(_pareto_cat) * 0.2))
_receita_top20p = _pareto_cat.head(_top20p_cats)["pct_receita"].sum()
_n_cats_total   = len(_pareto_cat)

# Concentração de produtos
_pareto_prod = (
    fe.groupby("id_produto")
    .agg(receita=("preco", "sum"))
    .reset_index()
    .sort_values("receita", ascending=False)
    .reset_index(drop=True)
)
_pareto_prod["pct_acum"] = (_pareto_prod["receita"] / _pareto_prod["receita"].sum() * 100).cumsum()
_total_produtos = len(_pareto_prod)
_n_prods_80     = (_pareto_prod["pct_acum"] <= 80).sum() + 1
_pct_prods_80   = _n_prods_80 / _total_produtos * 100

# Concentração de sellers
_pareto_seller = (
    fe.groupby("id_vendedor")
    .agg(receita=("preco", "sum"), n_periodos=("periodo", "nunique"))
    .reset_index()
    .sort_values("receita", ascending=False)
    .reset_index(drop=True)
)
_pareto_seller["pct_receita"] = _pareto_seller["receita"] / _pareto_seller["receita"].sum() * 100
_pareto_seller["pct_acum"]    = _pareto_seller["pct_receita"].cumsum()
_total_sellers  = len(_pareto_seller)
_n_sellers_80   = (_pareto_seller["pct_acum"] <= 80).sum() + 1
_pct_sellers_80 = _n_sellers_80 / _total_sellers * 100

# Sellers críticos
_n_trim_total = fe["periodo"].nunique()
_pareto_seller["pct_trim_ativo"] = _pareto_seller["n_periodos"] / _n_trim_total * 100
_criticos = _pareto_seller[
    (_pareto_seller["pct_receita"] >= 1.0) & (_pareto_seller["pct_trim_ativo"] >= 50)
]
_n_criticos           = len(_criticos)
_pct_receita_criticos = _criticos["pct_receita"].sum()

# Efeito Pareto no tempo
def _cats_80(grupo):
    g = grupo.groupby("nome_categoria_produto")["preco"].sum().sort_values(ascending=False)
    acum = (g / g.sum() * 100).cumsum()
    return (acum < 80).sum() + 1

_pareto_tempo = fe.groupby("periodo").apply(_cats_80).reset_index()
_pareto_tempo.columns = ["periodo", "n_cats_80"]
_pareto_tempo = _pareto_tempo.sort_values("periodo")
_delta_cats   = int(_pareto_tempo["n_cats_80"].iloc[-1]) - int(_pareto_tempo["n_cats_80"].iloc[0])
_direcao_pareto = "DILUIÇÃO (positivo)" if _delta_cats > 0 else "CONCENTRAÇÃO (risco)" if _delta_cats < 0 else "ESTÁVEL"

# Quadrante portfólio (Q1/Q2/Q4)
_periodos_ord = sorted(fe["periodo"].dropna().unique())
_evo = (
    fe.groupby(["periodo", "nome_categoria_produto"])
    .agg(receita=("preco", "sum"), frete=("valor_frete", "sum"))
    .reset_index()
)
_evo["pct_frete"] = _evo["frete"] / _evo["receita"] * 100
_cats_val = (
    _evo.groupby("nome_categoria_produto")["periodo"].nunique()
    .pipe(lambda s: s[s >= max(2, len(_periodos_ord) // 3)].index)
)
_m0 = _evo[_evo["periodo"] == _periodos_ord[0]].set_index("nome_categoria_produto")[["receita", "pct_frete"]]
_mf = _evo[_evo["periodo"] == _periodos_ord[-1]].set_index("nome_categoria_produto")[["receita", "pct_frete"]]
_comp = _m0.join(_mf, lsuffix="_i", rsuffix="_f", how="inner")
_comp = _comp[_comp.index.isin(_cats_val)].copy()
_comp["d_receita"] = (_comp["receita_f"] - _comp["receita_i"]) / _comp["receita_i"] * 100
_comp["d_frete"]   = _comp["pct_frete_f"] - _comp["pct_frete_i"]
_n_q1 = len(_comp[(_comp["d_receita"] >= 0) & (_comp["d_frete"] <= 0)])
_n_q2 = len(_comp[(_comp["d_receita"] >= 0) & (_comp["d_frete"] >  0)])
_n_q4 = len(_comp[(_comp["d_receita"] <  0) & (_comp["d_frete"] >  0)])

# Commoditização (ticket)
_top10_cats = _pareto_cat.head(10)["nome_categoria_produto"].tolist()
_ticket = (
    fe.groupby(["periodo", "nome_categoria_produto"])
    .agg(ticket_medio=("preco", "mean"))
    .reset_index()
)
_tpivot = (
    _ticket[_ticket["nome_categoria_produto"].isin(_top10_cats)]
    .pivot_table(index="nome_categoria_produto", columns="periodo", values="ticket_medio")
)
_tpivot["delta"] = _tpivot.iloc[:, -1] - _tpivot.iloc[:, 0]
_n_commodit  = (_tpivot["delta"] < 0).sum()
_n_agregando = (_tpivot["delta"] > 0).sum()


# Renovação de sellers (análise 9)
_seller_trim_v = (
    fe.groupby(["periodo", "id_vendedor"])["preco"].sum()
    .reset_index(name="receita")
)
_sellers_por_trim_v = _seller_trim_v.groupby("periodo")["id_vendedor"].apply(set).to_dict()
_periodos_v = sorted(_sellers_por_trim_v.keys())
_pct_novos_list = []
for _idx_v, _p_v in enumerate(_periodos_v):
    if _idx_v < 2:
        continue
    _ant_v = _sellers_por_trim_v[_periodos_v[_idx_v-1]] | _sellers_por_trim_v[_periodos_v[_idx_v-2]]
    _at_v  = _sellers_por_trim_v[_p_v]
    _pct_novos_list.append(len(_at_v - _ant_v) / len(_at_v) * 100 if _at_v else 0)
_pct_novos_medio_v = np.mean(_pct_novos_list) if _pct_novos_list else 0
s_renovacao = "✅ RENOVAÇÃO SAUDÁVEL" if _pct_novos_medio_v > 20 else "⚠️  BASE FOSSILIZADA"

# ─── Semáforos ────────────────────────────────────────────────────────────────
s_concentracao = "⚠️  RISCO"      if _receita_top20p > 80    else "✅ OK"
s_pareto       = "⚠️  CONCENTRANDO" if _delta_cats < 0       else "✅ DILUINDO"
s_armadilha    = "⚠️  ATENÇÃO"    if _n_q2 > 3               else "✅ OK"
s_sellers      = "⚠️  RISCO"      if _pct_sellers_80 < 10    else "✅ OK"
s_commodit     = "⚠️  ATENÇÃO"    if _n_commodit > _n_agregando else "✅ OK"
s_renovacao    = "⚠️  BASE FOSSILIZADA" if _pct_novos_medio_v <= 20 else "✅ RENOVAÇÃO SAUDÁVEL"

# ─── Sinal geral ──────────────────────────────────────────────────────────────
_n_alertas = sum([
    _receita_top20p > 80,
    _pct_sellers_80 < 10,
    _delta_cats < 0,
    _n_q2 > 3,
    _n_commodit > _n_agregando,
    _pct_novos_medio_v <= 20,
])
if _n_alertas == 0:
    sinal = "✅ MOTOR DE RECEITA ROBUSTO"
elif _n_alertas <= 2:
    sinal = "⚠️  ATENÇÃO — CONCENTRAÇÃO MODERADA COM RISCOS IDENTIFICÁVEIS"
else:
    sinal = "🔴 RISCO ESTRUTURAL — CONCENTRAÇÃO ELEVADA"

# ─── Impressão ────────────────────────────────────────────────────────────────
print("=" * 65)
print("SÍNTESE — BLOCO 2: QUALIDADE DO MOTOR DE RECEITA")
print("=" * 65)

print(f"""
[ CONCENTRAÇÃO ]
  Categorias para 80% da receita   : {_n_cats_80} de {_n_cats_total}
  Top 20% das categorias = receita  : {_receita_top20p:.1f}% — {s_concentracao}
  Produtos para 80% da receita     : {_n_prods_80} de {_total_produtos} ({_pct_prods_80:.1f}%)
  Sellers para 80% da receita      : {_n_sellers_80} de {_total_sellers} ({_pct_sellers_80:.1f}%) — {s_sellers}

[ DEPENDÊNCIA DE SELLERS ]
  Sellers críticos (≥1% receita + ≥50% trimestres ativos) : {_n_criticos}
  Receita concentrada nesses sellers críticos              : {_pct_receita_criticos:.1f}%

[ EFEITO PARETO NO TEMPO ]
  Cats para 80% no 1º trimestre    : {int(_pareto_tempo["n_cats_80"].iloc[0])}
  Cats para 80% no último trimestre: {int(_pareto_tempo["n_cats_80"].iloc[-1])}
  Variação                         : {_delta_cats:+d} — {_direcao_pareto} — {s_pareto}

[ QUALIDADE DO PORTFÓLIO ]
  Categorias Q1 (crescimento saudável)  : {_n_q1}
  Categorias Q2 (armadilha crescimento) : {_n_q2} — {s_armadilha}
  Categorias Q4 (deterioração total)    : {_n_q4}
  Categorias com ticket em queda        : {_n_commodit} de {len(_tpivot)} — {s_commodit}
  Categorias com ticket em alta         : {_n_agregando} de {len(_tpivot)}

[ RENOVAÇÃO DA BASE DE SELLERS ]
  % médio de sellers novos por trimestre: {_pct_novos_medio_v:.1f}% — {s_renovacao}
""")

print("=" * 65)
print("VEREDICTO PARCIAL DO BLOCO 2")
print("=" * 65)
print(f"""
Sinal geral: {sinal}

Próximo passo → Bloco 3: a operação suporta crescimento?
""")


---
*Próximo notebook: `03_EDA_escalabilidade_operacional.ipynb` — Crescer fortalece ou fragiliza o negócio?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
